In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# Original input files
BASE_DIR = "/content/drive/MyDrive/GSE149009"

clinical_path = os.path.join(
    BASE_DIR, "GSE149009_clean_clinical_metadata.csv"
)
expression_path = os.path.join(
    BASE_DIR, "GSE149009_gbm_rna_deseq2.txt"
)

# Check that both files exist
for path in [clinical_path, expression_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    print("Found:", path)

# Load the files
clinical = pd.read_csv(clinical_path)
expression = pd.read_csv(
    expression_path, sep="\t", index_col=0
)

print("Clinical metadata shape:", clinical.shape)
print("Expression matrix shape:", expression.shape)

display(clinical.head())
display(expression.iloc[:5, :5])


Mounted at /content/drive
Found: /content/drive/MyDrive/GSE149009/GSE149009_clean_clinical_metadata.csv
Found: /content/drive/MyDrive/GSE149009/GSE149009_gbm_rna_deseq2.txt
Clinical metadata shape: (68, 12)
Expression matrix shape: (37361, 68)


,patient_id,gsm_as_listed_in_dictionary,disease_state,age,gender,idh_mutation,survival_days,tissue,gsm,gsm_mapping_note,expr_column,survival_missing
0,G1,GSM4487293,GBM (IV),53.0,M,WT,22.0,GBM tumor tissue,GSM4487293,GEO Series roster sequence; OtB mapping cross-...,OtB6179,False
1,G2,GSM4487294,GBM (IV),69.0,M,NaN,25.0,GBM tumor tissue,GSM4487294,GEO Series roster sequence; OtB mapping cross-...,OtB6180,False
2,G3,GSM4487295,GBM (IV),36.0,M,WT,31.0,GBM tumor tissue,GSM4487295,GEO Series roster sequence; OtB mapping cross-...,OtB6181,False
3,G4,GSM4487296,GBM (IV),68.0,M,NaN,32.0,GBM tumor tissue,GSM4487296,GEO Series roster sequence; OtB mapping cross-...,OtB6182,False
4,G5,GSM4487297,GBM (IV),74.0,F,NaN,185.0,GBM tumor tissue,GSM4487297,GEO Series roster sequence; OtB mapping cross-...,OtB6183,False


,OtB6179,OtB6180,OtB6181,OtB6182,OtB6183
ENSG,,,,,
ENSG00000000003,5.828821,6.240547,6.959024,6.716199,7.711898
ENSG00000000005,3.602619,3.602619,3.602619,3.602619,3.602619
ENSG00000000419,4.687706,5.080394,5.480945,5.279210,3.602619
ENSG00000000457,6.172641,5.581485,6.391304,5.642001,8.708584
ENSG00000000460,6.013425,5.606976,7.124049,4.997947,10.533999


In [ ]:
from google.colab import drive
# ============================================================
# GBM SURVIVAL + GO PATHWAY ANALYSIS: FULL RECOVERY SCRIPT
# Dataset: GSE149009
#
# Rebuilds gene-survival correlations from the original files,
# corrects p-values for multiple testing, annotates Ensembl IDs,
# creates a unique-symbol preranked gene list, and runs GSEA.
#
# IMPORTANT:
# - This is an exploratory association analysis.
# - Survival days may be censored; Spearman correlation does not
#   replace a censoring-aware survival model.
# - GSEA results can differ from the earlier analysis because
#   duplicate gene symbols are resolved differently here.
# ============================================================

# ----------------------------
# 1. MOUNT GOOGLE DRIVE
# ----------------------------

from google.colab import drive
drive.mount('/content/drive')

# ----------------------------
# 2. INSTALL REQUIRED PACKAGES
# ----------------------------

!pip -q install mygene gseapy statsmodels scipy pandas numpy

# ----------------------------
# 3. IMPORT LIBRARIES
# ----------------------------

import os
import time
import warnings
import numpy as np
import pandas as pd
import mygene
import gseapy as gp

from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ----------------------------
# 4. DEFINE FILE LOCATIONS
# ----------------------------

DATA_DIR = '/content/drive/MyDrive/GSE149009'
OUT_DIR = os.path.join(DATA_DIR, 'recovered_full_analysis')

os.makedirs(OUT_DIR, exist_ok=True)

CLINICAL_FILE = os.path.join(
    DATA_DIR, 'GSE149009_clean_clinical_metadata.csv'
)

EXPRESSION_FILE = os.path.join(
    DATA_DIR, 'GSE149009_gbm_rna_deseq2.txt'
)

# Verify both source files exist before continuing.
for path in [CLINICAL_FILE, EXPRESSION_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f'File not found: {path}\n'
            'Check the filename and Google Drive folder.'
        )

print('Source files found:')
print(CLINICAL_FILE)
print(EXPRESSION_FILE)
print()

# ----------------------------
# 5. LOAD AND VALIDATE DATA
# ----------------------------

clinical = pd.read_csv(CLINICAL_FILE)

expression = pd.read_csv(
    EXPRESSION_FILE,
    sep='\t',
    index_col=0
)

print('Clinical metadata shape:', clinical.shape)
print('Expression matrix shape:', expression.shape)

required_columns = ['survival_days', 'expr_column']

missing_columns = [
    c for c in required_columns if c not in clinical.columns
]

if missing_columns:
    raise ValueError(
        f'Missing clinical columns: {missing_columns}'
    )

# Remove samples without survival information.
clinical_survival = clinical.dropna(
    subset=['survival_days']
).copy()

# Ensure the expression matrix contains the required samples.
samples = clinical_survival['expr_column'].astype(str).tolist()

missing_samples = [
    s for s in samples if s not in expression.columns
]

if missing_samples:
    raise ValueError(
        f'{len(missing_samples)} clinical samples were not found '
        f'in the expression matrix. Examples: {missing_samples[:10]}'
    )

# Convert expression and survival values to numeric.
X = expression.loc[:, samples].apply(
    pd.to_numeric, errors='coerce'
)

survival = pd.to_numeric(
    clinical_survival['survival_days'],
    errors='coerce'
).to_numpy(dtype=float)

# Check for missing/non-finite values.
if not np.isfinite(survival).all():
    raise ValueError(
        'Non-finite survival values remain after filtering.'
    )

if X.isna().any().any():
    raise ValueError(
        'Expression data contain missing or non-numeric values. '
        'Inspect the source matrix before continuing.'
    )

if len(samples) != 59:
    print(
        'WARNING: Expected 59 samples based on the previous run, '
        f'but found {len(samples)}.'
    )

print('Samples with survival data:', len(samples))
print('Genes in original matrix:', X.shape[0])
print('Survival days: median =',
      float(np.median(survival)),
      '| mean =', round(float(np.mean(survival)), 2))
print()

# Save a reproducibility summary.
audit = {
    'dataset': ['GSE149009'],
    'clinical_records': [len(clinical)],
    'expression_genes': [expression.shape[0]],
    'expression_samples': [expression.shape[1]],
    'samples_with_survival': [len(samples)],
    'mean_survival_days': [float(np.mean(survival))],
    'median_survival_days': [float(np.median(survival))]
}

pd.DataFrame(audit).to_csv(
    os.path.join(OUT_DIR, '00_audit_summary.csv'),
    index=False
)

# ----------------------------
# 6. SPEARMAN CORRELATION
#    FOR EACH NON-CONSTANT GENE
# ----------------------------

print('Calculating gene-survival correlations...')
print('This step may take a few minutes.')

rows = []
constant_genes = 0
invalid_genes = 0

for i, (gene, values) in enumerate(X.iterrows(), start=1):

    values = values.to_numpy(dtype=float)

    # Skip genes with no variation.
    if np.std(values) < 1e-10:
        constant_genes += 1
        continue

    rho, p_value = spearmanr(values, survival)

    if np.isfinite(rho) and np.isfinite(p_value):
        rows.append((str(gene), float(rho), float(p_value)))
    else:
        invalid_genes += 1

    if i % 5000 == 0:
        print(f'Processed {i:,} / {X.shape[0]:,} genes')

results = pd.DataFrame(
    rows,
    columns=['gene', 'spearman_rho', 'p_value']
)

if results.empty:
    raise RuntimeError(
        'No valid gene correlations were calculated.'
    )

# Benjamini-Hochberg FDR correction.
results['FDR'] = multipletests(
    results['p_value'].to_numpy(),
    method='fdr_bh'
)[1]

results = results.sort_values(
    'p_value'
).reset_index(drop=True)

results.to_csv(
    os.path.join(
        OUT_DIR,
        '01_full_gene_survival_associations.csv'
    ),
    index=False
)

# Full Ensembl-ID ranking, sorted by signed Spearman rho.
all_id_ranking = results[
    ['gene', 'spearman_rho']
].sort_values(
    'spearman_rho',
    ascending=False
)

all_id_ranking.to_csv(
    os.path.join(
        OUT_DIR,
        '02_prerank_all_Ensembl_IDs.csv'
    ),
    index=False
)

print()
print('GENE-LEVEL RESULTS')
print('------------------')
print('Genes tested:', len(results))
print('Constant genes removed:', constant_genes)
print('Invalid correlations skipped:', invalid_genes)
print('Genes with FDR < 0.05:',
      int((results['FDR'] < 0.05).sum()))
print()

# ----------------------------
# 7. ANNOTATE ENSEMBL IDS
#    USING MYGENE
# ----------------------------

print('Annotating gene IDs with MyGene.info...')

mg = mygene.MyGeneInfo()

gene_ids = results['gene'].astype(str).tolist()

# Remove version suffixes only for the annotation query.
# Preserve original IDs in the results table.
query_ids = [
    gene_id.split('.')[0] for gene_id in gene_ids
]

annotation_records = []

# Batch queries to improve reliability and allow retries.
BATCH_SIZE = 1000

for start in range(0, len(query_ids), BATCH_SIZE):

    batch = query_ids[start:start + BATCH_SIZE]

    for attempt in range(3):
        try:
            batch_results = mg.querymany(
                batch,
                scopes='ensembl.gene',
                fields='symbol,name',
                species='human',
                as_dataframe=False,
                returnall=False,
                verbose=False,
                dotfield=False
            )

            if isinstance(batch_results, dict):
                batch_results = batch_results.get(
                    'out', []
                )

            annotation_records.extend(batch_results)
            break

        except Exception as e:
            print(
                f'Annotation batch {start // BATCH_SIZE + 1} '
                f'failed (attempt {attempt + 1}/3): {e}'
            )

            if attempt == 2:
                raise RuntimeError(
                    'Gene annotation failed after 3 attempts. '
                    'Check the internet connection and rerun '
                    'this section.'
                )

            time.sleep(3 * (attempt + 1))

    completed = min(start + BATCH_SIZE, len(query_ids))
    print(f'Annotated-query progress: {completed:,} / {len(query_ids):,}')

ann_df = pd.DataFrame(annotation_records)

if ann_df.empty or 'query' not in ann_df.columns:
    raise RuntimeError(
        'MyGene returned no usable annotations. '
        'Check the service and try again.'
    )

# Keep only fields that actually exist in the response.
annotation_columns = ['query']

for col in ['symbol', 'name', 'notfound']:
    if col in ann_df.columns:
        annotation_columns.append(col)

ann_df = ann_df[annotation_columns].copy()

# Some IDs can produce multiple records. Retain one record per query.
ann_df = ann_df.drop_duplicates(
    subset=['query'],
    keep='first'
)

# Match results using version-stripped Ensembl IDs.
results_for_annotation = results.copy()
results_for_annotation['annotation_query'] = (
    results_for_annotation['gene']
    .astype(str)
    .str.split('.')
    .str[0]
)

annotated = results_for_annotation.merge(
    ann_df,
    left_on='annotation_query',
    right_on='query',
    how='left'
)

annotated = annotated.drop(
    columns=['query', 'annotation_query'],
    errors='ignore'
)

annotated.to_csv(
    os.path.join(
        OUT_DIR,
        '03_annotated_gene_survival_associations.csv'
    ),
    index=False
)

mapped_count = int(
    annotated['symbol'].notna().sum()
) if 'symbol' in annotated.columns else 0

print()
print('ANNOTATION RESULTS')
print('------------------')
print('IDs with symbols:', mapped_count)
print('IDs without symbols:', len(annotated) - mapped_count)
print()

if mapped_count == 0:
    raise RuntimeError(
        'No gene symbols were returned. Do not run GSEA '
        'until annotation has been fixed.'
    )

# ----------------------------
# 8. CREATE UNIQUE-SYMBOL
#    PRERANKED GENE LIST
# ----------------------------

print('Creating unique-symbol ranking...')

with_symbols = annotated.dropna(
    subset=['symbol']
).copy()

# Remove blank symbols.
with_symbols['symbol'] = (
    with_symbols['symbol'].astype(str).str.strip()
)

with_symbols = with_symbols[
    (with_symbols['symbol'] != '') &
    (with_symbols['symbol'].str.lower() != 'nan')
].copy()

with_symbols['abs_rho'] = (
    with_symbols['spearman_rho'].abs()
)

# Resolve duplicate symbols by retaining the gene with the
# strongest absolute correlation; use p-value to break ties.
unique_symbols = (
    with_symbols.sort_values(
        ['abs_rho', 'p_value'],
        ascending=[False, True]
    )
    .drop_duplicates(
        subset=['symbol'],
        keep='first'
    )
    .copy()
)

ranked = unique_symbols[
    ['symbol', 'spearman_rho', 'p_value', 'FDR']
].sort_values(
    'spearman_rho',
    ascending=False
)

ranked.to_csv(
    os.path.join(
        OUT_DIR,
        '04_prerank_unique_symbols.csv'
    ),
    index=False
)

print('Unique symbols supplied to GSEA:', len(ranked))
print()

if len(ranked) < 100:
    raise RuntimeError(
        'Unexpectedly few unique symbols were mapped. '
        'Inspect the annotation output before continuing.'
    )

# ----------------------------
# 9. RUN GO BIOLOGICAL PROCESS
#    PRERANKED GSEA
# ----------------------------

print('Running GO Biological Process preranked GSEA...')
print('Using 1,000 permutations and seed 42.')
print('This can take several minutes.')

gsea = gp.prerank(
    rnk=ranked[['symbol', 'spearman_rho']],
    gene_sets='GO_Biological_Process_2023',
    permutation_num=1000,
    min_size=15,
    max_size=500,
    seed=42,
    outdir=None,
    verbose=True
)

pathways = gsea.res2d.copy()

if pathways.empty:
    raise RuntimeError(
        'GSEA returned no pathways. Inspect the GSEA log '
        'and gene-set overlap.'
    )

pathways.to_csv(
    os.path.join(
        OUT_DIR,
        '05_GO_Biological_Process_prerank_REBUILT.csv'
    ),
    index=False
)

# Standardize FDR column to numeric.
fdr_column = 'FDR q-val'

if fdr_column not in pathways.columns:
    raise RuntimeError(
        f'Expected column {fdr_column!r} was not found. '
        f'Available columns: {list(pathways.columns)}'
    )

pathways[fdr_column] = pd.to_numeric(
    pathways[fdr_column],
    errors='coerce'
)

sig = pathways[
    pathways[fdr_column] < 0.05
].copy().sort_values(
    fdr_column,
    ascending=True
)

sig.to_csv(
    os.path.join(
        OUT_DIR,
        '06_significant_GO_pathways_FDR05.csv'
    ),
    index=False
)

# ----------------------------
# 10. PRINT TOP PATHWAYS
# ----------------------------

print()
print('=' * 65)
print('FINAL ANALYSIS SUMMARY')
print('=' * 65)

print('Samples included:', len(samples))
print('Genes tested:', len(results))
print('Constant genes removed:', constant_genes)
print('Genes with FDR < 0.05:',
      int((results['FDR'] < 0.05).sum()))
print('IDs with symbols:', mapped_count)
print('Unique symbols ranked:', len(ranked))
print('GO pathways tested:', len(pathways))
print('GO pathways with FDR < 0.05:', len(sig))
print()

if len(sig) > 0:
    print('TOP SIGNIFICANT GO PATHWAYS')
    print('---------------------------')

    display_columns = [
        c for c in [
            'Term',
            'NES',
            'NOM p-val',
            'FDR q-val',
            'FWER p-val'
        ] if c in sig.columns
    ]

    print(sig[display_columns].head(20).to_string(index=False))

else:
    print(
        'No GO pathways met the FDR < 0.05 threshold '
        'in this run.'
    )

# ----------------------------
# 11. SAVE A RUN SUMMARY
# ----------------------------

summary = pd.DataFrame([{
    'dataset': 'GSE149009',
    'survival_samples': len(samples),
    'original_expression_genes': X.shape[0],
    'genes_tested': len(results),
    'constant_genes_removed': constant_genes,
    'invalid_correlations_skipped': invalid_genes,
    'genes_FDR_below_005': int(
        (results['FDR'] < 0.05).sum()
    ),
    'IDs_with_symbols': mapped_count,
    'unique_symbols_ranked': len(ranked),
    'GO_pathways_tested': len(pathways),
    'GO_pathways_FDR_below_005': len(sig),
    'gene_sets': 'GO_Biological_Process_2023',
    'permutations': 1000,
    'seed': 42
}])

summary.to_csv(
    os.path.join(OUT_DIR, '07_final_run_summary.csv'),
    index=False
)

print()
print('ALL DONE.')
print('Output folder:')
print(OUT_DIR)
print()
print('FILES CREATED:')
for filename in sorted(os.listdir(OUT_DIR)):
    print(' -', filename)

print()
print(
    'Interpretation reminder: positive NES indicates enrichment '
    'toward genes positively correlated with observed survival; '
    'negative NES indicates enrichment toward genes negatively '
    'correlated with observed survival. These are associations, '
    'not evidence of causation.'
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.7/689.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 3.4 MB/s eta 0:00:00
Source files found:
/content/drive/MyDrive/GSE149009/GSE149009_clean_clinical_metadata.csv
/content/drive/MyDrive/GSE149009/GSE149009_gbm_rna_deseq2.txt

Clinical metadata shape: (68, 12)
Expression matrix shape: (37361, 68)
Samples with survival data: 59
Genes in original matrix: 37361
Survival days: median = 171.0 | mean = 875.49

Calculating gene-survival correlations...
This step may take a few minutes.
Processed 5,000 / 37,361 genes
Processed 10,000 / 37,361 genes
Processed 15,000 / 37,361 genes
Processed 20,000 / 37,361 genes
Processed 25,000 / 37,361 genes
Processed 30,000 / 37,361 genes

GENE-LEVEL RESULTS
------------------
Genes tested: 30617
Constant genes removed: 6744
Invalid correlations s

2026-10-13 08:22:12,352 [WARNING] Duplicated values found in preranked stats: 20.70% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-10-13 08:22:12,353 [INFO] Parsing data files for GSEA.............................


Unique symbols supplied to GSEA: 26709

Running GO Biological Process preranked GSEA...
Using 1,000 permutations and seed 42.
This can take several minutes.


2026-10-13 08:22:12,561 [INFO] Downloading and generating Enrichr library gene sets......
2026-10-13 08:22:12,775 [INFO] 2697 gene_sets have been filtered out when max_size=500 and min_size=15
2026-10-13 08:22:12,777 [INFO] 2710 gene_sets used for further statistical testing.....
2026-10-13 08:22:12,780 [INFO] Start to run GSEA...Might take a while..................
2026-10-13 08:29:48,505 [INFO] Congratulations. GSEApy runs successfully................




FINAL ANALYSIS SUMMARY
Samples included: 59
Genes tested: 30617
Constant genes removed: 6744
Genes with FDR < 0.05: 0
IDs with symbols: 27928
Unique symbols ranked: 26709
GO pathways tested: 2710
GO pathways with FDR < 0.05: 66

TOP SIGNIFICANT GO PATHWAYS
---------------------------
                                                                                             Term      NES  NOM p-val  FDR q-val  FWER p-val
                                                           Mitochondrial Translation (GO:0032543) 2.349539   0.000000   0.000000       0.000
                                                       Mitochondrial Gene Expression (GO:0140053) 2.295154   0.000000   0.000449       0.001
                                                                Chromatin Remodeling (GO:0006338) 2.160135   0.000000   0.004787       0.015
                                                      mRNA Splicing, Via Spliceosome (GO:0000398) 2.115475   0.000000   0.006283       0.032
         

In [ ]:
import os
import zipfile
from google.colab import files

folder = '/content/drive/MyDrive/GSE149009/recovered_full_analysis'
zip_path = '/content/GBM_GO_results.zip'

filenames = [
    '05_GO_Biological_Process_prerank_REBUILT.csv',
    '06_significant_GO_pathways_FDR05.csv'
]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for filename in filenames:
        path = os.path.join(folder, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f'Cannot find: {path}')
        z.write(path, arcname=filename)

files.download(zip_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/GSE149009/recovered_full_analysis/06_significant_GO_pathways_FDR05.csv'

df = pd.read_csv(path)
print(df.to_string(index=False))

   Name                                                                                              Term        ES       NES  NOM p-val  FDR q-val  FWER p-val   Tag % Gene %                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [ ]:
import pandas as pd

path = '/content/drive/MyDrive/GSE149009/GSE149009_clean_clinical_metadata.csv'
df = pd.read_csv(path)

print(df.columns.tolist())
print(df.head(3).to_string())

['patient_id', 'gsm_as_listed_in_dictionary', 'disease_state', 'age', 'gender', 'idh_mutation', 'survival_days', 'tissue', 'gsm', 'gsm_mapping_note', 'expr_column', 'survival_missing']
  patient_id gsm_as_listed_in_dictionary disease_state   age gender idh_mutation  survival_days            tissue         gsm                                                                  gsm_mapping_note expr_column  survival_missing
0         G1                  GSM4487293      GBM (IV)  53.0      M           WT           22.0  GBM tumor tissue  GSM4487293  GEO Series roster sequence; OtB mapping cross-checked against GEO sample records     OtB6179             False
1         G2                  GSM4487294      GBM (IV)  69.0      M          NaN           25.0  GBM tumor tissue  GSM4487294  GEO Series roster sequence; OtB mapping cross-checked against GEO sample records     OtB6180             False
2         G3                  GSM4487295      GBM (IV)  36.0      M           WT           31.0  GBM 

In [ ]:
import os

folder = "/content/drive/MyDrive/GSE149009"

for root, dirs, files in os.walk(folder):
    # Avoid listing every file in nested output folders
    depth = root.replace(folder, "").count(os.sep)
    if depth <= 2:
        for file in files:
            print(os.path.join(root, file))


/content/drive/MyDrive/GSE149009/GSE149009_gbm_rna_deseq2.txt
/content/drive/MyDrive/GSE149009/GSE149009_clean_clinical_metadata.csv
/content/drive/MyDrive/GSE149009/results/pca_coordinates.csv
/content/drive/MyDrive/GSE149009/results/expression_qc_summary.csv
/content/drive/MyDrive/GSE149009/results/survival/ranked_by_absolute_correlation.csv
/content/drive/MyDrive/GSE149009/results/survival/gene_survival_associations.csv
/content/drive/MyDrive/GSE149009/results/survival/week2_summary.csv
/content/drive/MyDrive/GSE149009/results/survival/annotated_gene_survival_associations.csv
/content/drive/MyDrive/GSE149009/results/survival/ranked_gene_symbols.csv
/content/drive/MyDrive/GSE149009/recovered_full_analysis/00_audit_summary.csv
/content/drive/MyDrive/GSE149009/recovered_full_analysis/01_full_gene_survival_associations.csv
/content/drive/MyDrive/GSE149009/recovered_full_analysis/02_prerank_all_Ensembl_IDs.csv
/content/drive/MyDrive/GSE149009/recovered_full_analysis/03_annotated_gene_sur

In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/GSE149009"

files_to_check = {
    "Audit summary": f"{base}/recovered_full_analysis/00_audit_summary.csv",
    "Final run summary": f"{base}/recovered_full_analysis/07_final_run_summary.csv",
    "Original survival summary": f"{base}/results/survival/week2_summary.csv",
    "Gene associations": f"{base}/recovered_full_analysis/01_full_gene_survival_associations.csv",
}

for name, path in files_to_check.items():
    print(f"\n{'='*65}\n{name}\n{'='*65}")
    if os.path.exists(path):
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        if len(df) <= 30:
            print(df.to_string(index=False))
        else:
            print(df.head(3).to_string(index=False))
    else:
        print("File not found:", path)



Audit summary
Shape: (1, 7)
Columns: ['dataset', 'clinical_records', 'expression_genes', 'expression_samples', 'samples_with_survival', 'mean_survival_days', 'median_survival_days']
  dataset  clinical_records  expression_genes  expression_samples  samples_with_survival  mean_survival_days  median_survival_days
GSE149009                68             37361                  68                     59          875.491525                 171.0

Final run summary
Shape: (1, 14)
Columns: ['dataset', 'survival_samples', 'original_expression_genes', 'genes_tested', 'constant_genes_removed', 'invalid_correlations_skipped', 'genes_FDR_below_005', 'IDs_with_symbols', 'unique_symbols_ranked', 'GO_pathways_tested', 'GO_pathways_FDR_below_005', 'gene_sets', 'permutations', 'seed']
  dataset  survival_samples  original_expression_genes  genes_tested  constant_genes_removed  invalid_correlations_skipped  genes_FDR_below_005  IDs_with_symbols  unique_symbols_ranked  GO_pathways_tested  GO_pathways_FDR

In [ ]:

!pip -q install GEOparse

import GEOparse

gse = GEOparse.get_GEO(
    geo="GSE149009",
    destdir="/content",
    silent=True
)

keywords = ["survival", "status", "death", "follow", "alive",
            "vital", "event", "censor", "days"]

for gsm_id, sample in gse.gsms.items():
    metadata = sample.metadata

    relevant = {}
    for key, values in metadata.items():
        combined = " ".join(map(str, values))
        if any(word in (key + " " + combined).lower()
               for word in keywords):
            relevant[key] = values

    if relevant:
        print(f"\n{gsm_id} | {metadata.get('title', [''])[0]}")
        for key, values in relevant.items():
            print(f"  {key}: {values}")



GSM4487293 | G1
  status: ['Public on Mar 02 2021']
  characteristics_ch1: ['disease state: GBM (IV)', 'age: 53', 'gender: M', 'idh_mutation: WT', 'survival (days): 22', 'tissue: GBM tumor tissue']

GSM4487294 | G2
  status: ['Public on Mar 02 2021']
  characteristics_ch1: ['disease state: GBM (IV)', 'age: 69', 'gender: M', 'idh_mutation: NA', 'survival (days): 25', 'tissue: GBM tumor tissue']

GSM4487295 | G3
  status: ['Public on Mar 02 2021']
  characteristics_ch1: ['disease state: GBM (IV)', 'age: 36', 'gender: M', 'idh_mutation: WT', 'survival (days): 31', 'tissue: GBM tumor tissue']

GSM4487296 | G4
  status: ['Public on Mar 02 2021']
  characteristics_ch1: ['disease state: GBM (IV)', 'age: 68', 'gender: M', 'idh_mutation: NA', 'survival (days): 32', 'tissue: GBM tumor tissue']

GSM4487297 | G5
  status: ['Public on Mar 02 2021']
  characteristics_ch1: ['disease state: GBM (IV)', 'age: 74', 'gender: F', 'idh_mutation: NA', 'survival (days): 185', 'tissue: GBM tumor tissue']

GSM

In [ ]:

import pandas as pd
import os

path = (
    "/content/drive/MyDrive/GSE149009/"
    "recovered_full_analysis/"
    "05_GO_Biological_Process_prerank_REBUILT.csv"
)

gsea = pd.read_csv(path)

print("Columns:", gsea.columns.tolist())
print("Total pathways:", len(gsea))

# Identify the relevant columns despite possible naming differences
fdr_col = next(
    (c for c in gsea.columns if c.lower() in ["fdr q-val", "fdr", "fdr_q_val"]),
    None
)
nes_col = next(
    (c for c in gsea.columns if c.lower() in ["nes", "normalized_enrichment_score"]),
    None
)

if fdr_col and nes_col:
    gsea[fdr_col] = pd.to_numeric(gsea[fdr_col], errors="coerce")
    gsea[nes_col] = pd.to_numeric(gsea[nes_col], errors="coerce")

    sig = gsea[gsea[fdr_col] < 0.05].copy()

    print("\nSignificant pathways (FDR < 0.05):", len(sig))
    print("Positive NES:", (sig[nes_col] > 0).sum())
    print("Negative NES:", (sig[nes_col] < 0).sum())

    print("\nTop 15 by FDR:")
    print(
        sig.sort_values(fdr_col)[
            [c for c in ["Term", "NES", "NOM p-val", "FDR q-val"]
             if c in sig.columns]
        ].head(15).to_string(index=False)
    )
else:
    print("Could not automatically identify FDR/NES columns.")
    print(gsea.head().to_string(index=False))


Columns: ['Name', 'Term', 'ES', 'NES', 'NOM p-val', 'FDR q-val', 'FWER p-val', 'Tag %', 'Gene %', 'Lead_genes']
Total pathways: 2710

Significant pathways (FDR < 0.05): 66
Positive NES: 59
Negative NES: 7

Top 15 by FDR:
                                                                                             Term      NES  NOM p-val  FDR q-val
                                                           Mitochondrial Translation (GO:0032543) 2.349539   0.000000   0.000000
                                                       Mitochondrial Gene Expression (GO:0140053) 2.295154   0.000000   0.000449
                                                                Chromatin Remodeling (GO:0006338) 2.160135   0.000000   0.004787
                                                      mRNA Splicing, Via Spliceosome (GO:0000398) 2.115475   0.000000   0.006283
                                                              Chromatin Organization (GO:0006325) 2.087810   0.000000   0.006540
     